# Select exactly K EEG sensors

This notebook runs `select_k_eeg_sensors.py` on the raw Siena EDF recordings. For each patient, it keeps the latest seizure/control pair(s) out of sensor selection, computes features independently for every electrode, and uses chronological forward selection plus one-for-one swap refinement to choose exactly **K** sensors.

> **Research only:** this retrospective analysis is not validated for diagnosis, treatment, seizure warning, or any other clinical use.

## 1. Settings

Change `K` and `PATIENT_IDS`, then run all cells. Use `PATIENT_IDS = None` to process every patient with at least two usable seizures. The first run reads raw EDF segments and creates a feature cache; later runs with the same preprocessing settings reuse it.

In [ ]:
# Number of EEG sensors to retain.
K = 10

# Example: ["PN00", "PN05"]. Use None for every eligible patient.
PATIENT_IDS = ["PN00"]

# The latest fraction of seizure events is kept out of sensor selection.
TEST_FRACTION = 0.20

# Swap refinement can repair mistakes made by greedy forward selection.
SWAP_REFINEMENT = True

# Set True only after changing preprocessing or if you want to ignore the cache.
FORCE_REBUILD_FEATURES = False

RANDOM_STATE = 42

## 2. Load the selector and raw-data manifest

In [ ]:
from dataclasses import asdict
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# This works when the notebook is launched from either the scripts folder or
# the analysis repository root.
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "select_k_eeg_sensors.py").exists():
    candidates = list(NOTEBOOK_DIR.rglob("select_k_eeg_sensors.py"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Run this notebook from the scripts directory or analysis repository root."
        )
    NOTEBOOK_DIR = candidates[0].parent

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import rolling_seizure_forecasting as forecast_workflow
import select_k_eeg_sensors as sensor_selection

if K < 1:
    raise ValueError("K must be at least 1.")
if not 0 < TEST_FRACTION < 1:
    raise ValueError("TEST_FRACTION must lie in (0, 1).")

paths = forecast_workflow.project_paths(NOTEBOOK_DIR)
paths["feature_cache"] = paths["processed"] / "fixed_k_sensor_selection"
forecast_config = forecast_workflow.ForecastConfig(test_fraction=TEST_FRACTION)
manifest = sensor_selection.load_selection_manifest(paths, forecast_config)

available_patients = sorted(
    manifest["patient_id"].astype(str).unique(),
    key=sensor_selection._natural_key,
)
requested_patients = (
    available_patients
    if PATIENT_IDS is None
    else list(dict.fromkeys(str(patient) for patient in PATIENT_IDS))
)
unknown_patients = sorted(set(requested_patients) - set(available_patients))
if unknown_patients:
    raise ValueError(
        f"Unknown patient(s): {', '.join(unknown_patients)}. "
        f"Available: {', '.join(available_patients)}"
    )

print(f"Raw EEG: {paths['raw']}")
print(f"K: {K}")
print(f"Patients: {', '.join(requested_patients)}")
print(f"Feature cache: {paths['feature_cache']}")

FileNotFoundError: Run this notebook from the scripts directory or analysis repository root.

## 3. Run fixed-K sensor selection

Only the earlier seizure events and their assigned interictal controls are used to rank sensors. The held-out latest seizure signal is not opened by the selector.

In [ ]:
result_rows = []
trace_frames = []
exclusion_rows = []

for patient_id in requested_patients:
    patient_manifest = manifest.loc[
        manifest["patient_id"].astype(str).eq(patient_id)
    ].copy()
    seizure_count = patient_manifest.loc[
        patient_manifest["episode_type"].eq("preictal"),
        "source_event_id",
    ].nunique()

    if seizure_count < 2:
        reason = "fewer than two usable seizures"
        exclusion_rows.append(
            {"patient_id": patient_id, "requested_k": K, "reason": reason}
        )
        print(f"{patient_id}: skipped ({reason}).")
        continue

    print(f"\n{patient_id}: selecting exactly {K} sensors...", flush=True)
    try:
        result, trace = sensor_selection.select_patient(
            patient_manifest,
            paths,
            k=K,
            test_fraction=TEST_FRACTION,
            random_state=RANDOM_STATE,
            swap_refinement=SWAP_REFINEMENT,
            force_rebuild_features=FORCE_REBUILD_FEATURES,
        )
    except Exception as error:
        reason = f"{type(error).__name__}: {error}"
        exclusion_rows.append(
            {"patient_id": patient_id, "requested_k": K, "reason": reason}
        )
        print(f"{patient_id}: skipped ({reason}).")
        continue

    row = asdict(result)
    row["selected_sensors"] = ", ".join(result.selected_sensors)
    result_rows.append(row)
    trace_frames.append(trace)
    print(f"{patient_id}: {row['selected_sensors']}")

results = pd.DataFrame(result_rows)
selection_trace = (
    pd.concat(trace_frames, ignore_index=True) if trace_frames else pd.DataFrame()
)
exclusions = pd.DataFrame(exclusion_rows)


PN00: selecting exactly 10 sensors...
PN00: loaded cached fixed-K features.
PN00: CP1, CP2, CZ, F9, F10, FC1, O1, O2, P3, T3


## 4. Selected sensors

In [ ]:
if results.empty:
    print("No patient produced a selection. Inspect the exclusions table below.")
else:
    display(
        results[
            [
                "patient_id",
                "requested_k",
                "selected_sensors",
                "available_sensor_count",
                "training_seizure_count",
                "held_out_seizure_count",
                "validation_method",
                "validation_auprc",
                "validation_brier",
            ]
        ].sort_values("patient_id")
    )

if not exclusions.empty:
    print("Excluded patients:")
    display(exclusions)

,patient_id,requested_k,selected_sensors,available_sensor_count,training_seizure_count,held_out_seizure_count,validation_method,validation_auprc,validation_brier
0,PN00,10,"CP1, CP2, CZ, F9, F10, FC1, O1, O2, P3, T3",29,4,1,expanding_chronological_validation,0.899138,0.066265


## 5. Accepted additions and swaps

This compact view shows how each final montage was constructed. The complete trace saved below also includes every rejected candidate.

In [ ]:
if not selection_trace.empty:
    accepted = selection_trace.loc[selection_trace["accepted"].astype(bool)].copy()
    display(
        accepted[
            [
                "patient_id",
                "phase",
                "step",
                "candidate_added",
                "candidate_removed",
                "candidate_sensors",
                "utility",
                "auprc",
                "brier",
            ]
        ]
    )

,patient_id,phase,step,candidate_added,candidate_removed,candidate_sensors,utility,auprc,brier
2,PN00,forward,1,CP1,,CP1,0.295782,0.353403,0.230483
53,PN00,forward,2,T3,,"CP1, T3",0.568274,0.606343,0.152278
68,PN00,forward,3,F10,,"CP1, F10, T3",0.752271,0.775758,0.093946
103,PN00,forward,4,O2,,"CP1, F10, O2, T3",0.822454,0.841253,0.075196
120,PN00,forward,5,F9,,"CP1, F9, F10, O2, T3",0.883508,0.899039,0.062122
135,PN00,forward,6,C3,,"C3, CP1, F9, F10, O2, T3",0.904907,0.919486,0.058313
168,PN00,forward,7,FC1,,"C3, CP1, F9, F10, FC1, O2, T3",0.906880,0.921594,0.058857
183,PN00,forward,8,CP2,,"C3, CP1, CP2, F9, F10, FC1, O2, T3",0.904839,0.920385,0.062185
204,PN00,forward,9,C4,,"C3, C4, CP1, CP2, F9, F10, FC1, O2, T3",0.894591,0.910135,0.062177
225,PN00,forward,10,CP5,,"C3, C4, CP1, CP2, CP5, F9, F10, FC1, O2, T3",0.876412,0.892726,0.065259


## 6. Save results

In [ ]:
output_dir = sensor_selection.DEFAULT_OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / f"selected_sensors_k{K}.csv"
trace_path = output_dir / f"selection_trace_k{K}.csv"
exclusions_path = output_dir / f"exclusions_k{K}.csv"
config_path = output_dir / f"selection_config_k{K}.json"

results.to_csv(results_path, index=False)
selection_trace.to_csv(trace_path, index=False)
exclusions.to_csv(exclusions_path, index=False)
config_path.write_text(
    json.dumps(
        {
            "k": K,
            "patients": requested_patients,
            "test_fraction": TEST_FRACTION,
            "random_state": RANDOM_STATE,
            "swap_refinement": SWAP_REFINEMENT,
            "force_rebuild_features": FORCE_REBUILD_FEATURES,
            "score": "AUPRC - 0.25 * Brier",
            "research_only": True,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print(f"Selections: {results_path}")
print(f"Full audit trace: {trace_path}")
print(f"Exclusions: {exclusions_path}")
print(f"Configuration: {config_path}")

Selections: C:\Users\shrey\OneDrive\Desktop\Cosmos\26-the-optimizers-analysis\final_project\results\fixed_k_sensor_selection\selected_sensors_k10.csv
Full audit trace: C:\Users\shrey\OneDrive\Desktop\Cosmos\26-the-optimizers-analysis\final_project\results\fixed_k_sensor_selection\selection_trace_k10.csv
Exclusions: C:\Users\shrey\OneDrive\Desktop\Cosmos\26-the-optimizers-analysis\final_project\results\fixed_k_sensor_selection\exclusions_k10.csv
Configuration: C:\Users\shrey\OneDrive\Desktop\Cosmos\26-the-optimizers-analysis\final_project\results\fixed_k_sensor_selection\selection_config_k10.json
